In [5]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd



# 1 Define input and output files


BASE_DIR = Path.cwd()

INPUT_PATH = (
    BASE_DIR
    / "clear_test_readabert_gaze_cognitive_rerun.csv"
)

OUTPUT_PATH = (
    BASE_DIR
    / "clear_text_states_phase9_rerun.csv"
)

STATE_PLOT = (
    BASE_DIR
    / "phase9_discretised_text_state_space_rerun.png"
)

print(
    "Input file exists:",
    INPUT_PATH.exists()
)

if not INPUT_PATH.exists():
    raise FileNotFoundError(
        f"Phase 8 output not found: {INPUT_PATH}"
    )



# 2 Load the combined difficulty and cognitive-effort data


state_df = pd.read_csv(
    INPUT_PATH
)

print(
    "\nShape:",
    state_df.shape
)

print(
    "\nInput sample:"
)

print(
    state_df[
        [
            "Excerpt",
            "BT.Easiness",
            "Predicted_BT_Easiness",
            "Difficulty_D",
            "Gaze_Score_Raw",
            "Cognitive_Effort_C"
        ]
    ].head()
)



# 3 Check variables required for state construction

required_columns = [
    "Excerpt",
    "BT.Easiness",
    "Predicted_BT_Easiness",
    "Difficulty_D",
    "Gaze_Score_Raw",
    "Cognitive_Effort_C"
]

missing_columns = [
    column
    for column in required_columns
    if column not in state_df.columns
]

if missing_columns:
    raise ValueError(
        "Missing required columns: "
        + ", ".join(
            missing_columns
        )
    )

if (
    state_df[
        [
            "Difficulty_D",
            "Cognitive_Effort_C"
        ]
    ]
    .isna()
    .any()
    .any()
):
    raise ValueError(
        "Missing D or C values were found."
    )

print(
    "\nRequired-column check passed."
)



# 4 Calculate empirical quantile thresholds for D and C


D_q1 = (
    state_df[
        "Difficulty_D"
    ]
    .quantile(
        1 / 3
    )
)

D_q2 = (
    state_df[
        "Difficulty_D"
    ]
    .quantile(
        2 / 3
    )
)

C_q1 = (
    state_df[
        "Cognitive_Effort_C"
    ]
    .quantile(
        1 / 3
    )
)

C_q2 = (
    state_df[
        "Cognitive_Effort_C"
    ]
    .quantile(
        2 / 3
    )
)

print(
    "\nDifficulty thresholds:"
)

print(
    "D q1:",
    D_q1
)

print(
    "D q2:",
    D_q2
)

print(
    "\nCognitive-effort thresholds:"
)

print(
    "C q1:",
    C_q1
)

print(
    "C q2:",
    C_q2
)



# 5 Discretise D and C into Low, Medium and High levels


state_df[
    "D_level"
] = (
    pd.qcut(
        state_df[
            "Difficulty_D"
        ],
        q=3,
        labels=[
            0,
            1,
            2
        ]
    )
    .astype(int)
)

state_df[
    "C_level"
] = (
    pd.qcut(
        state_df[
            "Cognitive_Effort_C"
        ],
        q=3,
        labels=[
            0,
            1,
            2
        ]
    )
    .astype(int)
)



# 6 Add readable labels


LEVEL_NAMES = {
    0: "Low",
    1: "Medium",
    2: "High"
}

state_df[
    "D_label"
] = (
    state_df[
        "D_level"
    ]
    .map(
        LEVEL_NAMES
    )
)

state_df[
    "C_label"
] = (
    state_df[
        "C_level"
    ]
    .map(
        LEVEL_NAMES
    )
)



# 7 Construct the two-component text state (D, C)


state_df[
    "Text_State"
] = list(
    zip(
        state_df[
            "D_level"
        ],
        state_df[
            "C_level"
        ]
    )
)

print(
    "\nExample discretised states:"
)

print(
    state_df[
        [
            "Difficulty_D",
            "D_level",
            "D_label",
            "Cognitive_Effort_C",
            "C_level",
            "C_label",
            "Text_State"
        ]
    ].head(20)
)




# 8 Check individual D and C level distributions


print(
    "\nD-level counts:"
)

print(
    state_df[
        "D_label"
    ]
    .value_counts()
    .reindex(
        [
            "Low",
            "Medium",
            "High"
        ]
    )
)

print(
    "\nC-level counts:"
)

print(
    state_df[
        "C_label"
    ]
    .value_counts()
    .reindex(
        [
            "Low",
            "Medium",
            "High"
        ]
    )
)



# 9 Examine the distribution of the nine text states


state_counts = (
    state_df
    .groupby(
        [
            "D_level",
            "C_level"
        ]
    )
    .size()
    .reset_index(
        name="Count"
    )
)

state_counts[
    "D_label"
] = (
    state_counts[
        "D_level"
    ]
    .map(
        LEVEL_NAMES
    )
)

state_counts[
    "C_label"
] = (
    state_counts[
        "C_level"
    ]
    .map(
        LEVEL_NAMES
    )
)

print(
    "\nText-state counts:"
)

print(
    state_counts[
        [
            "D_label",
            "C_label",
            "Count"
        ]
    ]
)

print(
    "\nNumeric state counts:"
)

print(
    state_df[
        "Text_State"
    ]
    .value_counts()
    .sort_index()
)

print(
    "\nNumber of observed text states:",
    state_df[
        "Text_State"
    ]
    .nunique()
)



# 10 Visualise the discretised D-C state space

plt.figure(
    figsize=(7, 6)
)

plt.scatter(
    state_df[
        "Difficulty_D"
    ],
    state_df[
        "Cognitive_Effort_C"
    ],
    alpha=0.5
)

plt.axvline(
    D_q1
)

plt.axvline(
    D_q2
)

plt.axhline(
    C_q1
)

plt.axhline(
    C_q2
)

plt.xlabel(
    "Readability Difficulty D"
)

plt.ylabel(
    "Cognitive Effort C"
)

plt.title(
    "Discretised Text-State Space"
)

plt.tight_layout()

plt.savefig(
    STATE_PLOT,
    dpi=300,
    bbox_inches="tight"
)

plt.close()



# 11 Save the quantile thresholds


thresholds_df = pd.DataFrame(
    [
        {
            "Signal": "D",
            "Low_Medium_Threshold": D_q1,
            "Medium_High_Threshold": D_q2
        },
        {
            "Signal": "C",
            "Low_Medium_Threshold": C_q1,
            "Medium_High_Threshold": C_q2
        }
    ]
)

THRESHOLD_PATH = (
    BASE_DIR
    / "phase9_discretisation_thresholds_rerun.csv"
)

thresholds_df.to_csv(
    THRESHOLD_PATH,
    index=False
)

print(
    "\nDiscretisation thresholds:"
)

print(
    thresholds_df
)



# 12 Export the final Phase 9 text-state dataset


state_df.to_csv(
    OUTPUT_PATH,
    index=False
)

print(
    "\nSaved:",
    OUTPUT_PATH
)

print(
    "Final shape:",
    state_df.shape
)



# 13 Confirm Phase 9 completion


print(
    "\nPHASE 9 COMPLETE"
)

print(
    "Text-state data:",
    OUTPUT_PATH
)

print(
    "Thresholds:",
    THRESHOLD_PATH
)

print(
    "State-space plot:",
    STATE_PLOT
)

Input file exists: True

Shape: (709, 24)

Input sample:
                                             Excerpt  BT.Easiness  \
0  And outside before the palace a great garden w...    -1.054013   
1  One day he had gone beyond any point which he ...    -1.238432   
2  So off went Lionel to be made a King. He had n...    -0.188186   
3  Father had been away in the country for three ...    -1.009999   
4  It was very dark at first, perhaps because the...    -1.258699   

   Predicted_BT_Easiness  Difficulty_D  Gaze_Score_Raw  Cognitive_Effort_C  
0              -1.732795      0.635710       -6.495265            0.566853  
1              -0.705971      0.443240      -26.958176            0.344768  
2               0.138993      0.284857      -35.851227            0.248251  
3              -0.657681      0.434188      -20.772287            0.411904  
4              -0.914117      0.482255      -21.697758            0.401859  

Required-column check passed.

Difficulty thresholds:
D q1: 0.389